In [0]:
landing_path = "/Volumes/workspace/default/ecommerce_data/landing/"
bronze_path = "/Volumes/workspace/default/ecommerce_data/bronze/"
checkpoint_path = "/Volumes/workspace/default/ecommerce_data/checkpoints/bronze/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("age", DoubleType(), True),
    StructField("gender", StringType(), True),
    StructField("country", StringType(), True),
    StructField("urban_rural", StringType(), True),
    StructField("income_level", DoubleType(), True),
    StructField("employment_status", StringType(), True),
    StructField("education_level", StringType(), True),
    StructField("relationship_status", StringType(), True),
    StructField("has_children", IntegerType(), True),
    StructField("household_size", IntegerType(), True),
    StructField("occupation", StringType(), True),
    StructField("ethnicity", StringType(), True),
    StructField("language_preference", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("weekly_purchases", IntegerType(), True),
    StructField("monthly_spend", DoubleType(), True),
    StructField("cart_abandonment_rate", DoubleType(), True),
    StructField("review_writing_frequency", IntegerType(), True),
    StructField("average_order_value", DoubleType(), True),
    StructField("preferred_payment_method", StringType(), True),
    StructField("coupon_usage_frequency", DoubleType(), True),
    StructField("loyalty_program_member", IntegerType(), True),
    StructField("referral_count", IntegerType(), True),
    StructField("product_category_preference", StringType(), True),
    StructField("shopping_time_of_day", StringType(), True),
    StructField("weekend_shopper", IntegerType(), True),
    StructField("impulse_purchases_per_month", IntegerType(), True),
    StructField("browse_to_buy_ratio", IntegerType(), True),
    StructField("return_frequency", IntegerType(), True),
    StructField("budgeting_style", StringType(), True),
    StructField("brand_loyalty_score", DoubleType(), True),
    StructField("impulse_buying_score", IntegerType(), True),
    StructField("environmental_consciousness", IntegerType(), True),
    StructField("health_conscious_shopping", IntegerType(), True),
    StructField("travel_frequency", DoubleType(), True),
    StructField("hobby_count", IntegerType(), True),
    StructField("social_media_influence_score", IntegerType(), True),
    StructField("reading_habits", IntegerType(), True),
    StructField("exercise_frequency", IntegerType(), True),
    StructField("stress_from_financial_decisions", IntegerType(), True),
    StructField("overall_stress_level", IntegerType(), True),
    StructField("sleep_quality", IntegerType(), True),
    StructField("physical_activity_level", IntegerType(), True),
    StructField("mental_health_score", IntegerType(), True),
    StructField("daily_session_time_minutes", DoubleType(), True),
    StructField("product_views_per_day", DoubleType(), True),
    StructField("ad_views_per_day", IntegerType(), True),
    StructField("ad_clicks_per_day", IntegerType(), True),
    StructField("wishlist_items_count", IntegerType(), True),
    StructField("cart_items_average", IntegerType(), True),
    StructField("checkout_abandonments_per_month", IntegerType(), True),
    StructField("purchase_conversion_rate", DoubleType(), True),
    StructField("app_usage_frequency", IntegerType(), True),
    StructField("notification_response_rate", IntegerType(), True),
    StructField("account_age_months", DoubleType(), True),
    StructField("last_purchase_date", StringType(), True),
    StructField("social_sharing_frequency", IntegerType(), True),
    StructField("premium_subscription", IntegerType(), True),
    StructField("return_rate", DoubleType(), True),
])

print("Schema defined with", len(schema.fields), "fields")

Schema defined with 60 fields


In [0]:
df_stream = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load(landing_path)
)

In [0]:
def consumer_print(batch_df, batch_id):
    row_count = batch_df.count()
    batch_df.write.mode("append").format("delta").save(bronze_path)
    print(f"[CONSUMER-LIVE] Micro-batch {batch_id}: consumed {row_count} rows")

query = (
    df_stream.writeStream
    .foreachBatch(consumer_print)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("[CONSUMER] All batches consumed!")

[CONSUMER] All batches consumed!


In [0]:
df_bronze_check = spark.read.format("delta").load(bronze_path)
print(df_bronze_check.count())
df_bronze_check.printSchema()

1020000
root
 |-- user_id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: double (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- cart_abandonment_rate: double (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- preferred_payment_method: string (nullable = true)
 |-- coupon_usage_fre